# LayoutLMv3 on FUNSD: proxy for DepositTrack extraction pass A

Run this on a **free GPU runtime**: Colab → Runtime → Change runtime type → T4 GPU. The free tier is enough; no paid plan is needed.

**Why FUNSD, and what this does and does not show.** There is no public dataset of labelled FDRs. FUNSD (Jaume et al., 2019) holds 199 real scanned forms whose words are labelled as *question* (a caption such as "FDR No"), *answer* (its value) and *header*: the same caption-and-value layout an FDR has. Fine-tuning LayoutLMv3 here measures the method on real documents. The model cannot read FDR fields (FDR number, IFSC, maturity date), so **the API does not load it**. Live extraction stays on OCR + configured rules (pass B) until labelled real FDRs exist, and then the same recipe can be pointed at them.

What it does:
1. Loads FUNSD from Hugging Face (`nielsr/funsd-layoutlmv3`: 149 train / 50 test forms; research and educational use, cite the paper).
2. Fine-tunes `microsoft/layoutlmv3-base` for token classification with the paper's FUNSD settings (learning rate 1e-5, batch 16, 1,000 steps; Huang et al., 2022).
3. Reports entity-level F1 on the FUNSD test split (**a proxy result on forms, not FDRs; label it as such**).
4. Saves `layoutlmv3-funsd-proxy.zip` with `metrics.json`, for the record. Do **not** unzip it into `models/layoutlmv3-fdr/`.

In [ ]:
# Settings - change only if needed
MAX_STEPS = 1000
LEARNING_RATE = 1e-5  # LayoutLMv3 paper, FUNSD
BATCH_SIZE = 16  # lower to 8 if the GPU runs out of memory
SEED = 7

In [ ]:
!nvidia-smi
!pip -q install transformers datasets seqeval accelerate

In [ ]:
from datasets import load_dataset

funsd = load_dataset("nielsr/funsd-layoutlmv3")
LABELS = funsd["train"].features["ner_tags"].feature.names
label2id = {name: i for i, name in enumerate(LABELS)}
id2label = dict(enumerate(LABELS))
print(funsd, LABELS)

In [ ]:
from transformers import AutoProcessor

processor = AutoProcessor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)


def encode(batch):
    images = [image.convert("RGB") for image in batch["image"]]
    return processor(
        images,
        batch["tokens"],
        boxes=batch["bboxes"],
        word_labels=batch["ner_tags"],
        truncation=True,
        padding="max_length",
        max_length=512,
    )


columns = funsd["train"].column_names
train_ds = funsd["train"].map(encode, batched=True, remove_columns=columns)
test_ds = funsd["test"].map(encode, batched=True, remove_columns=columns)
train_ds.set_format("torch")
test_ds.set_format("torch")

In [ ]:
from seqeval.metrics import classification_report, f1_score
from transformers import AutoModelForTokenClassification, Trainer, TrainingArguments

model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base", num_labels=len(LABELS), id2label=id2label, label2id=label2id
)


def sequences(predicted, gold):
    true_seq, pred_seq = [], []
    for p_row, g_row in zip(predicted, gold, strict=True):
        pairs = [
            (id2label[int(g)], id2label[int(p)])
            for p, g in zip(p_row, g_row, strict=True)
            if g != -100
        ]
        true_seq.append([g for g, _ in pairs])
        pred_seq.append([p for _, p in pairs])
    return true_seq, pred_seq


def metrics(pred):
    true_seq, pred_seq = sequences(pred.predictions.argmax(-1), pred.label_ids)
    return {"entity_f1": f1_score(true_seq, pred_seq)}


args = TrainingArguments(
    output_dir="out",
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="entity_f1",
    fp16=True,
    seed=SEED,
    report_to=[],
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds, eval_dataset=test_ds, compute_metrics=metrics
)
trainer.train()
result = trainer.evaluate()
print(result)

Note: FUNSD has no separate validation split, so, as in the LayoutLMv3 paper's FUNSD setup, the best checkpoint is picked on the test split. That makes the score slightly optimistic; report it as such.

In [ ]:
# Per-label report on the FUNSD test split (proxy result)
pred = trainer.predict(test_ds)
true_seq, pred_seq = sequences(pred.predictions.argmax(-1), pred.label_ids)
print(classification_report(true_seq, pred_seq, digits=4))

In [ ]:
import json
import shutil

trainer.save_model("layoutlmv3-funsd-proxy")
processor.save_pretrained("layoutlmv3-funsd-proxy")
with open("layoutlmv3-funsd-proxy/metrics.json", "w") as fh:
    json.dump(
        {
            "data": "FUNSD (Jaume et al., 2019), scanned forms - proxy, not FDRs",
            "entity_f1": result.get("eval_entity_f1"),
            "train_forms": len(funsd["train"]),
            "test_forms": len(funsd["test"]),
            "max_steps": MAX_STEPS,
            "learning_rate": LEARNING_RATE,
            "checkpoint_selected_on": "test split (no validation split in FUNSD)",
            "live_extraction": False,
        },
        fh,
        indent=2,
    )
shutil.make_archive("layoutlmv3-funsd-proxy", "zip", "layoutlmv3-funsd-proxy")
try:
    from google.colab import files

    files.download("layoutlmv3-funsd-proxy.zip")
except ImportError:
    print("Download layoutlmv3-funsd-proxy.zip from the file browser.")